# 第1章 Cosmos 3 Edge を動かしてみる (クイックスタート)

[nvidia/Cosmos3-Edge](https://huggingface.co/nvidia/Cosmos3-Edge) は、NVIDIA の世界モデル Cosmos 3 のうち、
Jetson などのエッジ機器向けの一番小さいモデル (4B パラメータ) です。1 つのモデルに 2 つの部分が入っています。

| 部分 | 入力 | 出力 | 使いみち |
| --- | --- | --- | --- |
| **Reasoner** | 文章・画像・動画 | 文章 (座標・時刻を含む) | 場面の理解、物の位置、作業の計画 |
| **Generator** | 文章・画像・行動 | 動画・行動 | 未来の予測、行動から映像 / 映像から行動 |

この章では、両方を 1 回ずつ動かして全体の感触をつかみます。

- **A. Reasoner**: ロボットの作業の画像を見て、作業の計画を立てさせる
- **B. Generator (画像 → 動画)**: 1 枚の画像から、その先の動画を作る
- **C. Generator (行動 → 動画)**: 車を「左に曲げたら」どう見えるかを予測する

各部分の詳しい使い方は第2章以降で 1 つずつ扱います。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外)。
A100 (40GB 以上) ではモデル全体を GPU に載せ、L4 (24GB) では使う部品だけを GPU に載せる設定 (CPU オフロード) に自動で切り替えます。

**安全フィルタ**: 生成には NVIDIA の安全フィルタ (`cosmos_guardrail`) を使います。使うには Hugging Face で
[nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) の利用申請をして承認を受け、
Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` を登録しておいてください。承認を待たずに試すときは、下の `GUARDRAIL` の印を外します。

In [ ]:
# @title GPU を確かめる
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    GPU_GB = 0
    print(f"CUDA の GPU が見つかりません ({DEVICE} で動かします。とても遅くなります)")

In [ ]:
# @title インストール (数分)
# Cosmos3OmniPipeline は Diffusers の main ブランチに入っている
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title 安全フィルタの設定とサンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

GUARDRAIL = True  # @param {type:"boolean"}
try:  # Colab のシークレットに HF_TOKEN があれば使う
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title 生成モデル (Generator) を読み込む関数
import gc
import json

from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageOps


def load_image(path, max_side=10_000):
    """画像を RGB で読む (max_side より大きければ縮める)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def load_pipe(flow_shift):
    """Cosmos3OmniPipeline を読み込む (2 回目からは読み込み済みのものを使う)。
    flow_shift は公式の推奨値: 画像→動画は 12.0、行動 (action) を使う生成は 10.0"""
    global pipe
    if "pipe" not in globals():
        pipe = Cosmos3OmniPipeline.from_pretrained(
            "nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
        if DEVICE == "cuda" and GPU_GB < 40:
            pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
        else:
            pipe.to(DEVICE)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift, use_karras_sigmas=False)
    return pipe


def free_gpu():
    """読み込んだモデルを GPU から降ろす"""
    for name in ("pipe", "model", "processor"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_video(path, width=640):
    display(Video(path, embed=True, width=width))

In [ ]:
# @title 質問用の関数を用意する
import json
import re

import av
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from transformers.video_utils import VideoMetadata

VIDEO_EXT = (".mp4", ".mov", ".webm", ".avi", ".mkv")


def load_image(path, max_side=1280):
    """画像を RGB で読む。大きすぎる画像はトークンが増えて遅くなるので縮める (座標は 0〜1000 の比率なので影響しない)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def read_video(path, fps=4, max_side=640, max_frames=256):
    """動画を fps 枚/秒に間引いて読む。返すのはコマの配列と、各コマの時刻を伝えるための情報 (VideoMetadata)。
    Reasoner は各コマに「<1.5 seconds>」のような時刻を付けて読むので、時刻を答える問題 (第4章) にはこの情報が欠かせない"""
    frames, t_next = [], 0.0
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < t_next:
                continue
            img = f.to_image()
            img.thumbnail((max_side, max_side))  # 1 コマのトークン数を抑える
            frames.append(np.asarray(img.convert("RGB")))
            t_next += 1 / fps
            if len(frames) >= max_frames:
                break
    n = len(frames)
    return np.stack(frames), VideoMetadata(total_num_frames=n, fps=fps, frames_indices=list(range(n)), duration=n / fps)


def ask(prompt, media=None, thinking=False, max_new_tokens=2048, fps=4, seed=0):
    """画像・動画 (なくてもよい) と質問を渡して、答えの文字列を返す。

    thinking=True のときは考えた過程 (<think>…</think>) を出してから答える。考えた過程は ask.last_think に残る。
    動画は fps 枚/秒で間引いて読む (推奨は 4)。
    """
    media = [] if media is None else media if isinstance(media, (list, tuple)) else [media]
    content, metadata = [], []
    for m in media:  # 画像・動画は質問文より前に置く (公式の推奨)
        if isinstance(m, str) and m.lower().endswith(VIDEO_EXT):
            frames, meta = read_video(m, fps=fps)
            content.append({"type": "video", "video": frames})
            metadata.append(meta)
        else:
            content.append({"type": "image", "image": load_image(m)})
    content.append({"type": "text", "text": prompt})
    # 動画は read_video で間引き済みなので、ここではもう間引かない
    video_kwargs = {"video_metadata": metadata, "do_sample_frames": False} if metadata else {}
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=thinking, **video_kwargs,
    ).to(model.device)
    # 公式のプロンプトガイドのサンプリング設定 (考える / 考えない で値が違う)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    torch.manual_seed(seed)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **sampling)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ask.last_think = ""
    if "</think>" in text:
        think, text = text.split("</think>", 1)
        ask.last_think = think.replace("<think>", "").strip()
    return text.strip()


def parse_json(text):
    """答えの中の JSON (```json … ``` で囲まれていてもよい) を取り出す。読めなければ None。
    小さなモデルの答えは、最後に余計な括弧が付く・真偽値の後ろに " が付く・最後に余計なカンマがある、
    といった崩れ方をするので、括弧の対応を数えて最初のまとまりだけを取り出し、よくある崩れを直してから読む"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    body = m.group(1) if m else text
    starts = [i for i in (body.find("["), body.find("{")) if i >= 0]
    if not starts:
        return None
    start = min(starts)
    depth, in_str, esc, end = 0, False, False, len(body)
    for k in range(start, len(body)):
        ch = body[k]
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                end = k + 1
                break
    chunk = body[start:end]
    fixed = re.sub(r'\b(true|false|null)"', r"\1", re.sub(r",\s*([\]}])", r"\1", chunk))
    for attempt in (chunk, fixed):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
    print("(JSON として読めませんでした)")
    return None


def load_reasoner():
    """Reasoner を読み込む (生成の章で、Generator と入れ替えて使うとき用)"""
    global model, processor
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", dtype=torch.bfloat16, device_map="auto")
    processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

## A. Reasoner: 画像を見て作業の計画を立てる

モデルカードのお手本です。Reasoner は既定で、考えた過程 (`<think>` … `</think>`) を出してから答えます。

In [ ]:
load_reasoner()
img = fetch(f"{HF}/example_reasoning_input.png")
display(IPImage(img, width=480))
answer = ask("The task is to put flower into the red bottle. Generate a plan consisting of subtasks for accomplish the task.",
             img, thinking=True, max_new_tokens=4096)
print("--- 考えた過程 (先頭) ---\n", ask.last_think[:1000])
print("\n--- 答え ---\n", answer)

## B. Generator: 1 枚の画像から動画を作る

Reasoner を GPU から降ろし、Generator を読み込みます。プロンプトは文章ではなく、決まった形の JSON (第7章で詳しく扱います) です。
お手本は 121 フレームですが、早く終わるよう 61 フレーム (24fps で約 2.5 秒) にしています。目安: A100 で数分、L4 ではその数倍。

In [ ]:
free_gpu()
pipe = load_pipe(flow_shift=12.0)
first_frame = load_image(fetch(f"{HF}/example_i2v_input.jpg"), max_side=10_000)
json_prompt = json.loads(Path(fetch(f"{HF}/example_i2v_prompt.json")).read_text())
negative_prompt = json.loads(Path(fetch(f"{HF}/negative_prompt.json")).read_text())
display(first_frame.resize((416, 240)))

result = pipe(
    prompt=json.dumps(json_prompt),
    negative_prompt=json.dumps(negative_prompt),
    image=first_frame,
    num_frames=61,
    height=480,
    width=832,
    fps=24.0,
    num_inference_steps=20,
    guidance_scale=6.0,
    generator=torch.Generator(device=DEVICE).manual_seed(0),
    add_resolution_template=False,
    add_duration_template=False,
)
export_to_video(result.video, "ch01_i2v.mp4", fps=24, macro_block_size=1)
show_video("ch01_i2v.mp4", 832)

## C. Generator: 行動を渡して未来を予測する

文章の代わりに**行動の数値** (ここでは車が左に曲がる 6 秒分の軌跡) を渡すと、
「この場所から車がこう動いたら、カメラにどう映るか」を予測した動画になります。これが世界モデルの中心的な使い方です (第8章)。

In [ ]:
from diffusers import CosmosActionCondition

RAW = "https://raw.githubusercontent.com/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
left_turn = json.loads(Path(fetch(f"{RAW}/generator/action/assets/actions/av_traj_left.json")).read_text())
start = load_image(fetch(f"{FW}/generator/action/assets/images/av_0.jpg"))
print(f"行動: {len(left_turn)} ステップ × {len(left_turn[0])} 個の数")

pipe = load_pipe(flow_shift=10.0)  # 行動を使う生成は flow_shift=10
result = pipe(
    prompt="You are an autonomous vehicle planning system.",
    action=CosmosActionCondition(
        mode="forward_dynamics",
        chunk_size=len(left_turn),
        domain_name="av",
        resolution_tier=480,
        raw_actions=torch.tensor(left_turn, dtype=torch.float32),
        image=start,
        view_point="ego_view",
    ),
    fps=10,
    num_inference_steps=30,
    guidance_scale=1.0,
    use_system_prompt=False,
    generator=torch.Generator(device=DEVICE).manual_seed(0),
)
export_to_video(result.video, "ch01_left_turn.mp4", fps=10, macro_block_size=1)
show_video("ch01_left_turn.mp4")

## 次の章へ

| 章 | 内容 |
| --- | --- |
| 2 | 画像と動画を言葉にする |
| 3 | 物の位置を座標で取る |
| 4 | 動画の「いつ」をつかむ |
| 5 | ロボットの作業を計画させる |
| 6 | 運転シーンを読み解く |
| 7 | 1 枚の画像から未来を動画にする |
| 8 | 車を動かして未来を見る |
| 9 | 映像から動きを逆算する |
| 10 | ロボットの手先を動かして未来を見る |
| 11 | 見て、動かす: ロボットの方策モデル |
| 12 | 考えて、想像して、確かめる |